# 01 · Demo del pipeline EcoStream (escenario B: comercio electrónico)
**Grupo 3** — Estructuras de Datos y Algoritmos 2026-2.

Este notebook ejecuta el pipeline de **10 etapas** sobre un stream de 200 000 eventos y muestra, para eventos
concretos, qué hace cada etapa y cómo su salida alimenta a la siguiente:
prioridades (E2), filtrado probabilístico (E6), similitud frente a métodos exactos (E7), análisis del grafo (E8)
y servicio al usuario final (E9).

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd
pd.set_option('display.max_colwidth', 80); pd.set_option('display.width', 160)

## 1. Generación del stream (esquema REES46 + componente sintético)

In [2]:
from ecostream.data.generator import make_catalog, generate_stream, StreamConfig, iter_events, EVENT_TYPES
cat = make_catalog(20_000, seed=42)
st = generate_stream(cat, StreamConfig(n_events=200_000, seed=42))
print(st.meta)
pd.DataFrame(list(__import__('itertools').islice(iter_events(st, cat), 0, 8)))

{'n_orders': 7000, 'n_lines': 15200, 'n_bursts': 4, 'n_dup': 4000, 'n_dirty': 969, 'n_users': 3333, 'blacklisted_orders': 29}


,event_id,event_time,event_type,product_id,category_code,brand,price,user_id,user_session,order_id,shipping,payment_token,device_id,premium,query
0,0,1.788221e+09,cart,13094.0,accessories.phone_case,huawei,79.46,1261,1261000,NaN,NaN,NaN,3784,False,None
1,1,1.788221e+09,view,7044.0,furniture.office.chair,ikea,924.58,849,849000,NaN,NaN,NaN,2548,False,None
2,2,1.788221e+09,view,5168.0,electronics.audio.headphone,motorola,257.59,1802,1802000,NaN,NaN,NaN,5406,False,None
3,4,1.788221e+09,view,2811.0,kids.toys,lego,137.78,1573,1573000,NaN,NaN,NaN,4720,False,None
4,3,1.788221e+09,view,581.0,computers.components.monitor,lenovo,934.19,862,862000,NaN,NaN,NaN,2587,False,None
5,5,1.788221e+09,cart,6601.0,electronics.audio.headphone,sony,259.23,3025,3025000,NaN,NaN,NaN,9076,False,None
6,6,1.788221e+09,payment_attempt,NaN,NaN,NaN,1641.80,2178,2178000,0.0,scheduled,2178.0,6534,False,None
7,7,1.788221e+09,purchase,17197.0,electronics.clocks,samsung,452.65,2178,2178000,0.0,scheduled,2178.0,6534,False,None


In [3]:
pd.Series(np.bincount(st.cols['event_type']), index=EVENT_TYPES, name='eventos').to_frame().T

,view,search,cart,remove_from_cart,purchase,payment_attempt,return,review
eventos,126667,16326,25449,5496,15509,7359,2657,4537


## 2. Ejecución de punta a punta con trazas

In [4]:
import importlib.util
spec = importlib.util.spec_from_file_location('demo', ROOT / 'scripts' / 'demo.py')
demo = importlib.util.module_from_spec(spec); spec.loader.exec_module(demo)
watch = demo.pick_watch(200_000, 20_000, 42)
from ecostream.pipeline import EcoStreamPipeline
p = EcoStreamPipeline(200_000, 20_000, 5000, refresh_every=20, seed=42,
                      watch={v for v in watch.values() if v is not None}, verbose=False).run()
watch

{'compra_express': 100036,
 'vista': 100009,
 'pago_fraudulento': 54074,
 'registro_sucio': 102391,
 'duplicado': 100441}

In [5]:
for name, eid in watch.items():
    print(f'\n▶ {name} (event_id={eid})')
    for stage, msg in p.tracer.log.get(eid, []):
        print(f'   [{stage}] {msg}')


▶ compra_express (event_id=100036)
   [E0] encolado en FIFO (posición 3698)
   [E0] desencolado en micro-lote de 5434
   [E1] validado/normalizado, único (búsqueda binaria en 96326 ids recientes), reordenado por tiempo
   [E2] insertado en heap con prioridad 2 (compra_express/premium); heap=6657
   [E2] extraído del heap tras esperar 0 ticks
   [E3] lookup O(1) en hash table vía LRU (hit-rate 52%), perfil de usuario e interacción en matriz dispersa actualizados; producto 'Teclado Samsung N158' (S/ 127.7)
   [E4] orden 3479 indexada en B+Tree (altura 2, 34 páginas de 4 KB)
   [E5] KD-Tree: 5 productos con atributos más cercanos [3485, 11002, 18125, 11146, 19228]
   [E6] orden en estado 'APROBADO' (su intento de pago pasó por Bloom + verificación exacta y CMS); compra contada en el CMS de tendencias (~1 en la última hora)
   [E7] compra aprobada alimenta los conjuntos de compradores; similares por comportamiento (LSH) [] y por contenido (HNSW) [647, 1569, 2689]
   [E8] orden 3479 entra 

## 3. Métricas por etapa

In [6]:
pd.DataFrame(p.stage_table())

,etapa,segundos,entradas,salidas,us_por_item,normalizado_tipo,normalizado_categoria,normalizado_precio,precio_faltante,rechazado_precio_negativo,...,pagos_bloqueado,alertas_velocidad,alertas_atendidas,lsh_reconstrucciones,productos_en_lsh,reconstrucciones,ordenes_en_ventana,aristas,modularidad,comunidades
0,E0 Ingesta (FIFO lista enlazada),1.6863,204000,204000,8.27,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,E1 Preprocesamiento (Merge Sort + búsqueda binaria),2.4573,204000,199827,12.05,196.0,173.0,220.0,195.0,176.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,E2 Cola de prioridades (min-heap),1.9263,199827,199831,9.64,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,E3 Hash table + LRU + matriz dispersa,1.6253,199827,199827,8.13,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,E4 B+Tree (índice en disco),0.0568,6999,6999,8.12,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,E5 KD-Tree (KNN exacto),12.4209,164173,164173,75.66,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,E6 Bloom Filter + Count-Min Sketch,1.3384,7202,7202,185.83,NaN,NaN,NaN,NaN,NaN,...,207.0,4.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,E7 MinHash+LSH y HNSW,90.3436,15104,15104,5981.43,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,3.0,14510.0,NaN,NaN,NaN,NaN,NaN
8,E8 Grafo co-compra: PageRank + Louvain,2.2255,0,0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3.0,1617.0,3360.0,0.8783,194.0
9,"E9 Servicio: Trie, Top-K, panel",1.2392,3,0,413051.43,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
s = p.summary()
pd.Series(s['espera_promedio_ticks'], name='espera promedio (ticks)').to_frame()

,espera promedio (ticks)
alerta,0.000
pago,0.000
compra_express/premium,0.000
compra_estandar,0.000
compra_programada,0.000
devolucion,0.000
carrito,0.000
resena,0.000
vista/busqueda,0.567


## 4. Filtrado probabilístico: Bloom (con verificación exacta) y Count-Min Sketch

In [8]:
m = p.s6.metrics.counters
print(f"Bloom: m={p.s6.bloom.m:,} bits, k={p.s6.bloom.k}; positivos={int(m['bloom_positivos'])}, "
      f"verificados={int(m['bloom_verificados'])}, falsos positivos={int(m['bloom_falsos_positivos'])}")
print('Órdenes:', s['ordenes'])
pd.DataFrame(p.s6.alerts)[['event_time', 'device_id', 'estimate', 'reason']]

Bloom: m=718,880 bits, k=10; positivos=205, verificados=205, falsos positivos=0
Órdenes: {'aprobado': 6964, 'bloqueado': 29, 'pendiente': 6}


,event_time,device_id,estimate,reason
0,1.788340e+09,10000002,16,velocidad: ~16 intentos en 5 min (umbral 15)
1,1.788831e+09,10000003,16,velocidad: ~16 intentos en 5 min (umbral 15); además token de pago en lista ...
2,1.788920e+09,10000001,16,velocidad: ~16 intentos en 5 min (umbral 15)
3,1.789863e+09,10000000,16,velocidad: ~16 intentos en 5 min (umbral 15); además token de pago en lista ...


## 5. Similitud aproximada vs exacta

In [9]:
import time
pid = int(max(p.s9.sales, key=p.s9.sales.get))
rows = []
for name, fa, fe in [('MinHash+LSH vs Jaccard exacto', p.s7.similar_behavior, p.s7.similar_behavior),
                     ('HNSW vs KNN exacto', p.s7.similar_content, p.s7.similar_content)]:
    t = time.perf_counter(); a = fa(pid, 10); ta = time.perf_counter() - t
    t = time.perf_counter(); e = fe(pid, 10, exact=True); te = time.perf_counter() - t
    rows.append({'método': name, 'ms aprox': ta * 1e3, 'ms exacto': te * 1e3,
                 'recall@10': len({x for x, _ in a} & {x for x, _ in e}) / max(1, len(e))})
print('Producto:', cat.names[pid]); pd.DataFrame(rows)

Producto: Licuadora Lg R336


,método,ms aprox,ms exacto,recall@10
0,MinHash+LSH vs Jaccard exacto,0.8650,1.8354,0.6
1,HNSW vs KNN exacto,1.1432,2.1087,1.0


## 6. Grafo de co-compra: PageRank vs grado, Louvain vs Label Propagation (mismo grafo)

In [10]:
from ecostream.graph.pagerank import degree_ranking
from ecostream.graph.communities import louvain, label_propagation, modularity, community_stats
g = p.s8.g; deg = degree_ranking(g)
top = np.argsort(-p.s8.pr)[:10]
display(pd.DataFrame({'producto': [cat.names[i] for i in top], 'PageRank': p.s8.pr[top], 'grado': deg[top]}))
act = g.degree > 0
res = []
for name, fn in [('Louvain', louvain), ('Label Propagation', label_propagation)]:
    t = time.perf_counter(); lab, _ = fn(g, seed=0); dt = time.perf_counter() - t
    res.append({'algoritmo': name, 'Q': modularity(g, lab), 'comunidades': community_stats(lab[act])['n_communities'], 's': dt})
pd.DataFrame(res)

,producto,PageRank,grado
0,Funda Huawei G324,0.003118,75.0
1,Televisor Jbl T224,0.002914,68.0
2,Licuadora Lg R336,0.002875,56.0
3,Refrigeradora Electrolux T554,0.002353,56.0
4,Microondas Samsung Z496,0.002208,50.0
5,Laptop Apple K499,0.001998,50.0
6,Smartwatch Xiaomi Z960,0.001816,41.0
7,Laptop Lg D108,0.001655,38.0
8,Juguete Lego W888,0.001448,31.0
9,Smartwatch Huawei C150,0.001408,34.0


,algoritmo,Q,comunidades,s
0,Louvain,0.878294,194,0.868357
1,Label Propagation,0.698937,410,0.104895


## 7. Servicio: autocompletado, recomendaciones Top-K, rango de precios y panel

In [11]:
for pref in ['smart', 'lap', 'zapa']:
    print(pref, '→', [n for _, n, _ in p.s9.autocomplete(pref, 4)])
uid = next(u for u, pr in p.s3.users.items() if len(pr['purchased']) >= 2)
print('compró:', [cat.names[x] for x in p.s3.user_profile(uid)['purchased'][-3:]])
pd.DataFrame([{'score': sc, 'producto': cat.names[q], 'fuentes': '+'.join(sorted(set(w)))} for sc, q, w in p.s9.recommend(uid, 8, explain=True)])

smart → ['Smartwatch Xiaomi Z960', 'Smartwatch Huawei C150', 'Smartphone Motorola U399', 'Smartphone Motorola P292']
lap → ['Laptop Apple K499', 'Laptop Lg D108', 'Laptop Logitech B593', 'Laptop Samsung A340']
zapa → ['Zapatillas Adidas F742', 'Zapatillas Adidas P858', 'Zapatillas Puma A463', 'Zapatillas Adidas K530']
compró: ['Casaca Adidas O903', 'Casaca Levis W135', 'Casaca Levis X386']


,score,producto,fuentes
0,0.553526,Casaca Adidas E898,LSH+PPR
1,0.490231,Casaca Levis O558,LSH+PPR
2,0.384265,Casaca Nike F552,LSH
3,0.354621,Casaca Puma K229,LSH
4,0.321287,Smartwatch Huawei C150,LSH+PPR
5,0.282748,Casaca Levis I108,LSH
6,0.255499,Casaca Puma K830,LSH
7,0.193670,Casaca Levis J707,HNSW+LSH


In [12]:
from ecostream.dashboard import build_dashboard
build_dashboard(p, str(ROOT / 'docs' / 'dashboard_notebook.html'))

'D:\\Maestria UTEC\\Clases\\Ciclo 2\\Estructura de datos y algoritmos\\Proyecto\\proyecto-eda-grupo3\\docs\\dashboard_notebook.html'